# 01 · Taxi series (C1-H2..H4, H8, H11)

Eyeball evidence for the zone-186 (Penn Station/Madison Sq West) metered-fare series, the window, and weather. Run from the repo root with the backend venv (`pip install -e 'backend[data,notebooks]'`). Reads tracked fixtures only.

In [ ]:
import sys; sys.path.insert(0, 'backend')
import pandas as pd, matplotlib.pyplot as plt
from app import config
from app.data import features, taxi, weather, sports

df = features.build('taxi')
print(config.TAXI_ZONE_ID, config.WINDOW_START, config.WINDOW_END, config.TRAIN_END)
df.head()

## H8: window per source

In [ ]:
srcs = {'taxi': taxi.load()['date'], 'weather_msg': weather.load_weather(offline=True).daily['date'],
        'weather_station': weather.load_weather(*config.SETTLEMENT_STATION_LATLON, offline=True).daily['date']}
for t in sports.TEAMS:
    srcs[f'schedule_{t}'] = pd.to_datetime(sports.read_schedule(t)['date']).dt.date
pd.DataFrame({k: [v.min(), v.max(), len(v)] for k, v in srcs.items()}, index=['min', 'max', 'rows']).T

## H2: revenue over time (weekly seasonality, holiday dips, no flatlines / 10x spikes)

In [ ]:
ts = df.set_index(pd.to_datetime(df['date']))['revenue']
ax = ts.plot(figsize=(14, 4), lw=0.6, label='daily')
ts.rolling(28, center=True).mean().plot(ax=ax, lw=2, label='28d mean')
ax.scatter(ts.index[df['is_holiday'].to_numpy()], ts[df['is_holiday'].to_numpy()], c='red', s=12, label='holiday', zorder=3)
ax.axvline(pd.Timestamp(config.HOLDOUT_START), c='k', ls='--', lw=1)
ax.legend(); ax.set_ylabel('USD metered fares'); plt.show()
df.nsmallest(8, 'revenue')[['date', 'revenue', 'is_holiday', 'rain', 'tmax']]

In [ ]:
df.boxplot(column='revenue', by='dow', figsize=(8, 4)); plt.suptitle(''); plt.title('revenue by day of week (Mon=0)'); plt.show()

## H3: MSG event nights (Knicks/Rangers home games)

In [ ]:
tr = df[df['split'] == 'train']
print(tr.groupby(['dow', 'home_game'])['revenue'].mean().unstack().assign(lift=lambda t: t[True] / t[False] - 1).round(3))

## H11: weather sanity and the rain-flag gap (for C3 rule 4)

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
d = df.set_index(pd.to_datetime(df['date']))
d['tmax'].plot(ax=ax[0]); ax[0].set_ylabel('tmax °F')
weather.load_weather(offline=True).daily.set_index(pd.to_datetime(d.index))['precipitation_sum'].plot(ax=ax[1], kind='line'); ax[1].set_ylabel('precip in')
plt.show()
print('rain share', round(df['rain'].mean(), 3), '| rain_open_hours share', round(df['rain_open_hours'].mean(), 3))
print('share of rain days with rain_open_hours', round(df.loc[df['rain'], 'rain_open_hours'].mean(), 3))

## Rain direction (H16 for taxi): expect rain days >= dry days on weekdays

In [ ]:
print(tr.groupby(['dow', 'rain'])['revenue'].mean().unstack().assign(ratio=lambda t: t[True] / t[False]).round(3))
s = features.add_settlement(df)
print('p̂ settle_rain (train)', round(features.event_frequency(s, 'settle_rain'), 3), '| MSG vs station agree', round((s['rain'] == s['settle_rain']).mean(), 3))